In [4]:
import pandas as pd
import os
import getpass
from sqlalchemy import create_engine, text
import kagglehub
import shutil
import os
import zipfile
import sys
from pathlib import Path
from sqlalchemy.engine import URL
from pathlib import Path

sys.path.append(str(Path().resolve().parents[0]))

from src.create_database import create_database

print('Done')

Done


In [5]:
cache_path = kagglehub.dataset_download("mashlyn/online-retail-ii-uci", force_download=True)

print("Downloaded to cache:", cache_path)

project_dir = "../data/01_raw"
os.makedirs(project_dir, exist_ok=True)

if cache_path.endswith(".zip"):
    with zipfile.ZipFile(cache_path, 'r') as z:
        z.extractall(project_dir)
    print("Extracted to:", project_dir)
else:
    for filename in os.listdir(cache_path):
        shutil.copy(os.path.join(cache_path, filename), project_dir)
    print("Copied to:", project_dir)

100%|██████████| 14.5M/14.5M [00:04<00:00, 3.33MB/s]

Extracting files...


Downloaded to cache: /home/zxrcodev/.cache/kagglehub/datasets/mashlyn/online-retail-ii-uci/versions/3
Copied to: ../data/01_raw


In [6]:
path_csv = '../data/01_raw/online_retail_II.csv'
df = pd.read_csv(path_csv)
df = df.dropna(subset=['Customer ID'])
df['Customer ID'] = df['Customer ID'].astype(int)

pd_to_csv_path = os.path.join('..', 'data', '02_to_sql', 'retail_to_sql.csv')

os.makedirs(os.path.dirname(pd_to_csv_path), exist_ok=True)
df.to_csv(pd_to_csv_path, index=False)

print(f"Saved to: {pd_to_csv_path}")

Saved to: ../data/02_to_sql/retail_to_sql.csv


In [7]:
db_name = input(str('Enter your database name: '))
password = getpass.getpass("Enter your database password: ")

base_url = URL.create(
    drivername="postgresql+psycopg2",
    username="postgres",
    password=password,
    host="localhost",
    port=5432
)

create_database(base_url, db_name)

engine = create_engine(base_url.set(database=db_name))

df.to_sql(
    'retail_records', 
    engine, 
    if_exists='replace'
)

Database 'retail' created successfully.


364